# DSL v3 + Gmsh 演示 — 左屏命令簿

按幻灯片顺序排列。每段 = 一个 slide。配 `dslv3_gmsh_presentation.html`/PDF 一起看。

环境: `C:\ProgramData\anaconda3\envs\metal-env\python.exe` (Py 3.11 + qiskit-metal + gmsh + meshio)。

## 0. setup — sys.path 自举

`src/qiskit_metal/` 自动加到 path。`_YAML_DIR` / `_SCRIPTS_DIR` / `_OUT_DIR` 供后面引用。

In [10]:
import os, sys
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
from pathlib import Path

_HERE = Path.cwd()
_REPO = next(p for p in [_HERE, *_HERE.parents] if (p / 'src' / 'quantum_dsl').exists())
_SRC = _REPO / 'src'
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

_DSL_DIR     = _REPO / 'examples' / 'dsl'
_YAML_DIR    = _DSL_DIR / 'yaml'
_SCRIPTS_DIR = _DSL_DIR / 'scripts'
_OUT_DIR     = _DSL_DIR / 'outputs'
_OUT_DIR.mkdir(parents=True, exist_ok=True)

import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
_f, _a = plt.subplots(figsize=(0.1, 0.1)); _a.plot([0, 1], [0, 1]); plt.close(_f); del _f, _a

print('repo :', _REPO)
print('python:', sys.version.split()[0])

repo : d:\BaiduSyncdisk\vsCOde\circuit\qiskit\qiskit-metal-worktrees\dyk07-main
python: 3.11.15


## Slide 2 — schema 顶层 keys

`src/qiskit_metal/toolbox_metal/dsl/schema.py:40-91`

In [11]:
from quantum_dsl.dsl.schema import (
    CURRENT_SCHEMA, ROOT_KEYS, GMSH_SIM_KEYS,
    LAYER_STACK_ENTRY_KEYS, MESH_KEYS,
)
print('CURRENT_SCHEMA :', CURRENT_SCHEMA)
print('ROOT_KEYS      :', sorted(ROOT_KEYS))
print('GMSH_SIM_KEYS  :', sorted(GMSH_SIM_KEYS))
print('LAYER_STACK    :', sorted(LAYER_STACK_ENTRY_KEYS))
print('MESH_KEYS      :', sorted(MESH_KEYS))

CURRENT_SCHEMA : qiskit-metal/design-dsl/3
ROOT_KEYS      : ['circuit', 'geometry', 'hamiltonian', 'netlist', 'schema', 'simulation', 'templates', 'vars']
GMSH_SIM_KEYS  : ['airbox', 'layer_stack', 'mesh', 'output', 'ports', 'symmetry']
LAYER_STACK    : ['eps_r', 'kind', 'material', 'tan_delta', 'thickness', 'z']
MESH_KEYS      : ['conductor_refine', 'max_size', 'max_size_jj', 'min_size']


## Slide 4 — build_ir 主流水线 (primitive-native YAML)

`yaml/chain_2q_native.metal.yaml` → `builder.py:393-488`

In [12]:
from quantum_dsl.dsl import build_ir

ir = build_ir(_YAML_DIR / 'chain_2q_native.metal.yaml')
print('schema     :', ir.schema)
print('components :', [c.name for c in ir.components])
print('vars       :', ir.vars)
print('circuit    :', ir.circuit)
print('netlist    :', ir.netlist)

schema     : qiskit-metal/design-dsl/3
components : ['Q1', 'Q2', 'bus']
vars       : {'qx': '1.2mm', 'pad_w': '420um', 'pad_h': '90um', 'trace_w': '12um', 'trace_gap': '7um', 'bus_attach': '0.34mm', 'c_q': '65fF', 'ej_q': '18GHz'}
circuit    : {'Q1': {'type': 'transmon', 'C': '65fF', 'pad_width': '420um'}, 'Q2': {'type': 'transmon', 'C': '65fF', 'pad_width': '420um'}, 'bus': {'width': '12um', 'gap': '7um'}}
netlist    : {'connections': [{'from': 'Q1.bus', 'to': 'bus.start'}, {'from': 'bus.end', 'to': 'Q2.bus'}]}


In [13]:
# 一个 PrimitiveIR 长什么样 (ir.py:21-37)
p = ir.components[0].primitives[0]
print('component :', p.component)
print('name      :', p.name)
print('kind      :', p.kind, '| shape:', p.shape)
print('subtract  :', p.subtract, '| layer:', p.layer)
print('geometry  :', p.geometry, ' (shapely; bounds in mm)')
print('bounds    :', p.geometry.bounds)

component : Q1
name      : pad_left
kind      : poly | shape: rectangle
subtract  : False | layer: 1
geometry  : POLYGON ((-1.5299999999999998 -0.045, -1.1099999999999999 -0.045, -1.1099999999999999 0.045, -1.5299999999999998 0.045, -1.5299999999999998 -0.045))  (shapely; bounds in mm)
bounds    : (-1.5299999999999998, -0.045, -1.1099999999999999, 0.045)


## Slide 5 — transmon_pocket 模板继承链

`yaml/transmon_pocket_2q.metal.yaml` → `component_templates.py:25-134` → `template_registry.py:69-74`

In [14]:
from quantum_dsl.dsl.template_registry import (
    BUILTIN_COMPONENT_TEMPLATE_PATHS, ComponentTemplateRegistry,
)
for k, v in BUILTIN_COMPONENT_TEMPLATE_PATHS.items():
    print(f'{k:20s} → dsl_templates/{v}')

qcomponent           → dsl_templates/core\qcomponent.yaml
base_qubit           → dsl_templates/core\base_qubit.yaml
transmon_pocket      → dsl_templates/qubits\transmon_pocket.yaml


In [15]:
reg = ComponentTemplateRegistry()
chain = reg.inheritance_chain('transmon_pocket')
print('inheritance_chain (qcomponent → ... → transmon_pocket):')
for t in chain:
    print(f'  - {t.id:20s}  extends={t.extends!r}')

inheritance_chain (qcomponent → ... → transmon_pocket):
  - qcomponent            extends=None
  - base_qubit            extends='qcomponent'
  - transmon_pocket       extends='base_qubit'


In [16]:
# 跑 transmon_pocket demo — 模板真的展开了, 导出的还是 NativeComponent
from quantum_dsl.dsl import build_design
design_tp = build_design(_YAML_DIR / 'transmon_pocket_2q.metal.yaml')
for name in ['Q1', 'Q2']:
    meta = design_tp.components[name].metadata['template']
    cls  = design_tp.components[name].__class__.__name__
    print(f'{name}: inherited={meta["inherited"]}  →  {cls}')

Q1: inherited=['qcomponent', 'base_qubit', 'transmon_pocket']  →  NativeComponent
Q2: inherited=['qcomponent', 'base_qubit', 'transmon_pocket']  →  NativeComponent


## Slide 6 — YAML 顶层 (chain_2q_native)

直接打开 `yaml/chain_2q_native.metal.yaml`。

In [17]:
yaml_text = (_YAML_DIR / 'chain_2q_native.metal.yaml').read_text(encoding='utf-8')
print(yaml_text[:1400])

# Hamiltonian-Circuit-Netlist-Geometry full-chain 示例。
#
# 和 native_2q_minimal.metal.yaml 相比，这个文件多展示了:
#
#   1. hamiltonian 引用 circuit 参数
#   2. circuit 统一定义 qubit 和 bus 的电路参数
#   3. netlist 用两个连接把 Q1 -> bus -> Q2 串起来
#   4. geometry.templates + $extend 复用 primitive 模板
#   5. derived metadata 中会出现 bus path length、pin middle、连接信息
#
# 运行:
#   C:\ProgramData\anaconda3\envs\metal-env\python.exe examples\dsl\run_chain_demo.py
#
schema: qiskit-metal/design-dsl/3

# 全局变量。建议把“会被多处引用”的尺寸和电路参数放这里。
vars:
  qx: 1.2mm        # Q1/Q2 离原点距离
  pad_w: 420um     # qubit pad 宽度
  pad_h: 90um      # qubit pad 高度
  trace_w: 12um    # bus/path/pin 宽度
  trace_gap: 7um   # pin gap
  bus_attach: 0.34mm  # qubit 局部坐标里 bus pin 到中心的 x 偏移
  c_q: 65fF        # circuit 层 qubit capacitance
  ej_q: 18GHz      # Hamiltonian 层 EJ

# hamiltonian 可以引用 circuit。这里 Q1/Q2 的 C 来自 circuit.Q*.C。
# 当前 DSL 不做物理求解，hamiltonian 会作为 resolved metadata 保留。
hamiltonian:
  subsystems:
    Q1: {model: transmon, EJ: "${ej_q}", C: "${circuit.

## Slide 7 — ${...} 表达式

`expression.py:91-108` (substitute_string) · `:140-195` (_eval_ast)

In [18]:
from quantum_dsl.dsl.expression import (
    walk_substitute, substitute_string, evaluate_expression,
)
ctx = {'vars': {'qx': 1.2, 'bus_attach': 0.34}, 'circuit': {'bus': {'width': 0.012}}}

# 整字段 = ${...}  →  原对象类型 (float)
print(repr(substitute_string('${circuit.bus.width}', ctx)))

# 字段含其它字符  →  字符串化
print(repr(substitute_string('-${vars.qx} + ${vars.bus_attach}', ctx)))

# 受限 AST: +-*/
print(evaluate_expression('vars.qx - vars.bus_attach', ctx))

0.012
'-1.2 + 0.34'
0.8599999999999999


## Slide 8 — DesignIR.derived

`ir.py:74-101` · `parsers/circuit.py:61-115`

In [19]:
import pprint
pprint.pp(ir.derived['circuit']['geometry']['Q1'])

{'bounds': [-1.6, -0.26, -0.7999999999999999, 0.26],
 'primitives': {'pad_left': {'kind': 'poly',
                             'shape': 'rectangle',
                             'bounds': [-1.5299999999999998,
                                        -0.045,
                                        -1.1099999999999999,
                                        0.045]},
                'pad_right': {'kind': 'poly',
                              'shape': 'rectangle',
                              'bounds': [-1.29, -0.045, -0.87, 0.045]},
                'pocket': {'kind': 'poly',
                           'shape': 'rectangle',
                           'bounds': [-1.6, -0.26, -0.7999999999999999, 0.26]},
                'jj': {'kind': 'junction',
                       'shape': 'line',
                       'bounds': [-1.2, -0.045, -1.2, 0.045]}},
 'pins': {'bus': {'points': [[-0.8599999999999999, -0.006],
                             [-0.8599999999999999, 0.006]],
                  'midd

In [20]:
pprint.pp(ir.derived['netlist'])

{'connections': [{'from': {'component': 'Q1', 'pin': 'bus'},
                  'to': {'component': 'bus', 'pin': 'start'},
                  'net_id': None},
                 {'from': {'component': 'bus', 'pin': 'end'},
                  'to': {'component': 'Q2', 'pin': 'bus'},
                  'net_id': None}]}


## Slide 9 — build_design (左路)

`builder.py:584-592` → `export_ir_to_metal builder.py:512-581`

In [21]:
design = build_design(_YAML_DIR / 'chain_2q_native.metal.yaml')
for table in ('poly', 'path', 'junction'):
    print(f'{table:10s}: {len(design.qgeometry.tables[table])} rows')
print('net_info rows:', len(design.net_info))
print('NativeComponent? ', design.components['Q1'].__class__.__name__)

poly      : 6 rows
path      : 2 rows
junction  : 2 rows
net_info rows: 4
NativeComponent?  NativeComponent


In [22]:
design.qgeometry.tables['poly'][['component', 'name', 'subtract', 'layer']]

,component,name,subtract,layer
0,1,pad_left,False,1
1,1,pad_right,False,1
2,1,pocket,True,1
3,2,pad_left,False,1
4,2,pad_right,False,1
5,2,pocket,True,1


In [23]:
design.net_info

,net_id,component_id,pin_name
0,1,1,bus
1,1,3,start
2,2,3,end
3,2,2,bus


## Slide 10 — Gmsh 入门: 最小 OCC 调用

`gmsh.model.occ.addBox` + `mesh.generate(3)`

In [24]:
import gmsh
gmsh.initialize()
gmsh.model.add('demo_box')
tag = gmsh.model.occ.addBox(0, 0, 0, 1, 1, 1)
gmsh.model.occ.synchronize()
print('box dimtag : (3,', tag, ')')
print('3D entities:', gmsh.model.getEntities(dim=3))
gmsh.model.mesh.generate(3)
print('node count :', len(gmsh.model.mesh.getNodes()[0]))
gmsh.finalize()

box dimtag : (3, 1 )
3D entities: [(3, 1)]
node count : 339


## Slide 12 — adapter 依赖白名单 (硬约束)

`_gmsh_*.py` 不能 import `designs.*` / `qlibrary.*` / `QGmshRenderer` / `LayerStackHandler`。

In [25]:
import re, pathlib
deny = re.compile(r'from\s+qiskit_metal\.(designs|qlibrary)|QGmshRenderer|LayerStackHandler|BoundsForPathAndPolyTables')
adapter_dir = _SRC / 'quantum_dsl' / 'dsl'
for f in sorted(adapter_dir.glob('_gmsh_*.py')) + [adapter_dir / 'gmsh_adapter.py']:
    hits = [(i+1, ln.rstrip()) for i, ln in enumerate(f.read_text(encoding='utf-8').splitlines()) if deny.search(ln)]
    flag = 'OK' if not hits else 'VIOLATION'
    print(f'{flag:9s} {f.name:25s}  ({len(hits)} hits)')
    for ln_no, ln in hits:
        print(f'         L{ln_no}: {ln}')

VIOLATION _gmsh_geometry.py          (10 hits)
         L5: `QDesign` / `QGmshRenderer` / `LayerStackHandler`。允许 import:
         L13: `qiskit_metal.toolbox_metal.layer_stack_handler`, `BoundsForPathAndPolyTables`,
         L14: 整个 `QGmshRenderer` 类, `renderer_base`。
         L41: # 内部 tracker (替代 QGmshRenderer.paths_dict / polys_dict / juncs_dict / ...)
         L100:     # fragment 后的 dimtag 重映射 (替代 QGmshRenderer:899-912 的内联 zip 循环)
         L258:     """画 JJ 矩形, 留 2D surface 在 layer 中心 z (照 `QGmshRenderer:467` 语义)。"""
         L279:     # 与 QGmshRenderer 一样根据 v1-v3 / v1-v4 距离选 winding
         L290:     # JJ 放在 layer 中心: 沿 z 平移 thickness/2 (与 QGmshRenderer:508 一致)
         L355:     与 `QGmshRenderer.add_endcaps(open_pins=...)` 的默认语义对齐 — endcap
         L560: # 计算 chip XY bounding box (替代 BoundsForPathAndPolyTables)
VIOLATION _gmsh_layers.py            (1 hits)
         L11: `LayerStackHandler`, `BoundsForPathAndPolyTables`, `QGmshRenderer`, `renderer_base`。
OK        _gmsh_mesh.py  

## Slide 14 — _sanitize 行为

`_gmsh_physical.py:62-73`

In [26]:
from quantum_dsl.dsl._gmsh_physical import _sanitize, PHYSICAL_GROUP_NAMING
for raw in ('Q1.bus.pad_left', '2um_pad', 'gnd layer 1', ''):
    print(f'{raw!r:25s}  →  {_sanitize(raw)!r}')
print()
for k, tpl in PHYSICAL_GROUP_NAMING.items():
    print(f'  {k:20s} {tpl}')

'Q1.bus.pad_left'          →  'Q1_bus_pad_left'
'2um_pad'                  →  'g_2um_pad'
'gnd layer 1'              →  'gnd_layer_1'
''                         →  'unnamed'

  ground_volume        gnd_layer{layer}
  ground_surface       gnd_layer{layer}_sfs
  substrate_volume     substrate_layer{layer}
  component_volume     {component}_{primitive}
  component_surface    {component}_{primitive}_sfs
  junction_surface     {component}_{primitive}_jj
  vacuum_volume        vacuum
  vacuum_outer         vacuum_outer
  port_lumped          port_{component}_{pin}
  port_ground          port_{component}_{pin}_gnd
  symmetry_surface     symmetry_{plane}


## Slide 15 — 端到端 build_mesh (右路)

`gmsh_adapter.py:355-467`. 写出 `outputs/chain_2q.msh`, 17 个 named physical groups。

In [27]:
from quantum_dsl.dsl.gmsh_adapter import build_mesh

mesh_path = _OUT_DIR / 'chain_2q.msh'
demo_mesh = {'max_size': 2.0, 'min_size': 0.5, 'max_size_jj': 0.5,
             'conductor_refine': {'min_dist': 1.0, 'max_dist': 3.0}}
result = build_mesh(
    _YAML_DIR / 'chain_2q_native.metal.yaml',
    output_path=mesh_path,
    options={'mesh': demo_mesh},
    show_gui=False,
)
print('mesh file        :', result.mesh_path)
print('mesh size (bytes):', result.mesh_path.stat().st_size)
print('physical groups  :', len(result.physical_groups))
for name, (dim, tags) in sorted(result.physical_groups.items()):
    print(f'  - {name} (dim={dim}, n_tags={len(tags)})')

mesh file        : d:\BaiduSyncdisk\vsCOde\circuit\qiskit\qiskit-metal-worktrees\dyk07-main\examples\dsl\outputs\chain_2q.msh
mesh size (bytes): 101768
physical groups  : 17
  - Q1_jj_jj (dim=2, n_tags=1)
  - Q1_pad_left (dim=3, n_tags=2)
  - Q1_pad_left_sfs (dim=2, n_tags=11)
  - Q1_pad_right (dim=3, n_tags=2)
  - Q1_pad_right_sfs (dim=2, n_tags=11)
  - Q2_jj_jj (dim=2, n_tags=1)
  - Q2_pad_left (dim=3, n_tags=2)
  - Q2_pad_left_sfs (dim=2, n_tags=11)
  - Q2_pad_right (dim=3, n_tags=2)
  - Q2_pad_right_sfs (dim=2, n_tags=11)
  - bus_center_trace (dim=3, n_tags=1)
  - bus_center_trace_sfs (dim=2, n_tags=8)
  - gnd_layer1 (dim=3, n_tags=1)
  - gnd_layer1_sfs (dim=2, n_tags=20)
  - substrate_layer3 (dim=3, n_tags=1)
  - vacuum (dim=3, n_tags=1)
  - vacuum_outer (dim=2, n_tags=60)


In [28]:
# meshio 回读 .msh — 第三方工具消费
import meshio
m = meshio.read(result.mesh_path)
print('nodes      :', len(m.points))
print('cell blocks:', len(m.cells))
print('field_data :', len(m.field_data), 'groups (= physical groups)')


nodes      : 385
cell blocks: 84
field_data : 17 groups (= physical groups)


开gui
`C:\ProgramData\anaconda3\envs\metal-env\python.exe examples\dsl\scripts\run_chain_gmsh_demo.py --gui`

### Slide 15 (备选) — 直接调 demo 脚本

`scripts/run_chain_gmsh_demo.py:102-107` 末尾有 6 个 required group 的 assertion。

In [29]:
import subprocess, sys
subprocess.run([sys.executable, str(_SCRIPTS_DIR / 'run_chain_gmsh_demo.py'),
                '--output', str(_OUT_DIR / 'chain_2q_via_script.msh')],
               check=True)

CompletedProcess(args=['c:\\ProgramData\\anaconda3\\envs\\metal-env\\python.exe', 'd:\\BaiduSyncdisk\\vsCOde\\circuit\\qiskit\\qiskit-metal-worktrees\\dyk07-main\\examples\\dsl\\scripts\\run_chain_gmsh_demo.py', '--output', 'd:\\BaiduSyncdisk\\vsCOde\\circuit\\qiskit\\qiskit-metal-worktrees\\dyk07-main\\examples\\dsl\\outputs\\chain_2q_via_script.msh'], returncode=0)

### Slide 15 (GUI) — 看 physical groups

PowerShell 跑 (本 notebook 别跑 GUI, 会抢 Qt event loop):

```
C:\ProgramData\anaconda3\envs\metal-env\python.exe examples\dsl\scripts\run_chain_gmsh_demo.py --gui
```

## Slide 16 — 单位防呆

`gmsh_adapter.py:170-191` (_MESH_LENGTH_MIN_MM / _MAX / _check_mesh_length_mm)

In [30]:
# 错误用法: 把 SI 米传 kwarg → 在 1e-5 mm 以下被 raise 拦截
try:
    build_mesh(_YAML_DIR / 'chain_2q_native.metal.yaml',
               options={'mesh': {'max_size': 0.000005}})
except ValueError as exc:
    print('OK — raised:', exc)

OK — raised: simulation.gmsh.mesh.max_size=5e-06 mm is outside the sane range [1e-05 mm, 100.0 mm]. mesh kwarg 单位 = mm float (与 IR simulation.gmsh.mesh.* 同语义); 若想表示 SI 米数值, 请乘以 1000 (e.g. 5e-6 米 → 0.005 mm)。参考 examples/dsl/.note/gmsh_walkthrough.md §5.2.


## Slide 17 — 跑测试

`tests/test_design_dsl_gmsh.py::test_m3_required_physical_groups`

In [31]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pytest',
                str(_REPO / 'tests' / 'test_design_dsl_gmsh.py') +
                '::test_m3_required_physical_groups', '-v'],
               cwd=_REPO, check=False)

CompletedProcess(args=['c:\\ProgramData\\anaconda3\\envs\\metal-env\\python.exe', '-m', 'pytest', 'd:\\BaiduSyncdisk\\vsCOde\\circuit\\qiskit\\qiskit-metal-worktrees\\dyk07-main\\tests\\test_design_dsl_gmsh.py::test_m3_required_physical_groups', '-v'], returncode=1)